# Demo 02 : create_agent

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · create-agent

We build the same loop from a model and a tool list.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

tools = [game_info, assets_for_game, search_design_docs]
question = {"messages": [("human", "Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, "
                                   "then the style guide section on the rule that applies. Which asset is at fault, and what should the Artist change?")]}


def show(result):
    for message in result["messages"]:
        print(message.type, str(getattr(message, "tool_calls", None) or message.content)[:150].replace("\n", " "))
    print("model calls:", sum(m.type == "ai" for m in result["messages"]))
    print("input tokens per call:", [m.usage_metadata["input_tokens"] for m in result["messages"] if m.type == "ai"])

We supply the same tools to `create_agent`. The return value is a compiled LangGraph graph, and `invoke` returns its final state.

In [3]:
from langchain.agents import create_agent

agent = create_agent(llm, tools=tools,
                     system_prompt="You are the studio assistant of Comet Arcade. Use the tools. Answer in three sentences.")
result = agent.invoke(question, {"recursion_limit": 16})
show(result)
print("answer:", result["messages"][-1].content)
print("graph type:", type(agent).__name__)
print("nodes:", list(agent.get_graph().nodes))
print("tool schema:", game_info.args)

human Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, then the style guide section on the rule that applies
ai [{'name': 'game_info', 'args': {'title': 'Comet Snake'}, 'id': 'call_CHzCzzxGKvv9BDd077fRdhD3', 'type': 'tool_call'}]
tool {"game_id": "CS-01", "title": "Comet Snake", "genre": "arcade", "status": "in development", "pitch": "Snake in space : eat stars, dodge comets, every 
ai [{'name': 'assets_for_game', 'args': {'game_id': 'CS-01'}, 'id': 'call_fwuYKKmAeAzrjBnPZLaDSFka', 'type': 'tool_call'}]
tool [{"asset_id": "CS-SPR-01", "kind": "sprite", "name": "snake head", "colour": "#7cf5c4", "status": "final"}, {"asset_id": "CS-SPR-02", "kind": "sprite"
ai [{'name': 'search_design_docs', 'args': {'query': 'style guide rule sprite contrast visibility against background comet asset colour contrast'}, 'id':
tool # Comet Arcade style guide ## Contrast  Anything that can end the game needs a contrast of at least 4.5 to 1 against the backgro

## b · agent-as-a-node

We put the compiled agent inside a parent graph. Shared message state passes into and out of the agent node. After the agent answers, a plain function writes the note for the Artist, so that step does not depend on the model.

In [4]:
from langgraph.graph import END, START, MessagesState, StateGraph


class StudioState(MessagesState):
    handover: str


def handover(state: StudioState):
    return {"handover": "For the Artist : " + str(state["messages"][-1].content)}


parent = StateGraph(StudioState)
parent.add_node("studio_agent", agent)
parent.add_node("handover", handover)
parent.add_edge(START, "studio_agent")
parent.add_edge("studio_agent", "handover")
parent.add_edge("handover", END)
out = parent.compile().invoke(question)
print("handover:", out["handover"])

handover: For the Artist : The at-fault asset is **CS-SPR-03, the comet sprite**: its draft blue `#3a4a7a` is too close to the deep-space background `#0b1030`, making the hazard hard to see. The style guide says game-ending hazards need at least **4.5:1 contrast** and comets must use a warm danger colour. The Artist should keep the comet’s shape and change its colour to the studio danger colour, **`#ff6b5e`**.


## c · custom-agent-state

We extend `AgentState` with a game id and a model call counter. A model hook adds the game id to the prompt, and a tool reads state through `ToolRuntime`. The runtime argument stays out of the tool schema.

In [5]:
from langchain.agents import AgentState
from langchain.agents.middleware import after_model, dynamic_prompt
from langchain.tools import ToolRuntime
from typing_extensions import NotRequired


class GameState(AgentState):
    game_id: str
    model_calls: NotRequired[int]


@dynamic_prompt
def game_prompt(request):
    return f"You assist the team of game {request.state['game_id']}. Use open_bugs for the bug list."


@after_model(state_schema=GameState)
def count_calls(state, runtime):
    return {"model_calls": state.get("model_calls", 0) + 1}


@tool
def open_bugs(runtime: ToolRuntime) -> str:
    """List the open bugs of the current game, with severity."""
    return json.dumps(rows("SELECT bug_id, title, severity FROM bugs WHERE game_id = ? AND status = 'open'",
                           runtime.state["game_id"]))


game_agent = create_agent(llm, tools=[open_bugs], state_schema=GameState,
                          middleware=[game_prompt, count_calls])
out = game_agent.invoke({"game_id": "CS-01", "messages": [
    ("human", "Which open bug of my game should the team fix first? Use the tool.")]})
print("game:", out["game_id"])
print("model calls:", out["model_calls"])
print("tool schema:", open_bugs.args)
print("answer:", out["messages"][-1].content)

game: CS-01
model calls: 2
tool schema: {}
answer: Fix **BUG-101: “Comet hard to see on the dark background”** first. It’s the only blocker, so it takes priority over the major and minor bugs.


## d · a-crew-as-tools

The Producer is one `create_agent`. Its tools are the crew : each crew tool makes one fresh model call that sees only what the tool passes in. The Designer picks a twist from the menu, the Coder writes the logic block, and the tests run real code without a model. The code stays on a shared board, so the Producer never reads it. We set `max_tokens`, because one uncapped reply can run for minutes.

In [6]:
import importlib.util
from pathlib import Path

kit = Path(data_folder("game-studio/kit"))
spec = importlib.util.spec_from_file_location("harness", kit / "harness.py")
harness = importlib.util.module_from_spec(spec)
spec.loader.exec_module(harness)

TEMPLATE = (kit / "template.html").read_text(encoding="utf-8")
RULES = (kit / "rules.md").read_text(encoding="utf-8")
TWISTS = json.loads((kit / "twists.json").read_text(encoding="utf-8"))
BRIEFS = json.loads((kit / "briefs.json").read_text(encoding="utf-8"))

In [7]:
from langchain.messages import HumanMessage, SystemMessage

crew_llm = make_llm(max_tokens=8000)
MENU = "\n".join(f"- {key} : {value['name']}. {value['pitch']} " + " ".join(value["rules"]) for key, value in TWISTS.items())
board = {"logic": harness.logic_of(TEMPLATE), "report": None, "attempts": 0, "twist": None, "order": [], "read": {}}


def code_block(text):
    blocks = re.findall(r"```(?:javascript|js)?\s*\n(.*?)```", text, re.S)
    found = max(blocks, key=len) if blocks else text
    return found.replace("/* LOGIC:START */", "").replace("/* LOGIC:END */", "").strip()


@tool
def ask_designer(brief: str) -> str:
    """Send a brief to the Designer. Returns the chosen twist and its rules."""
    board["order"].append("designer")
    reply = crew_llm.invoke([
        SystemMessage("You are the Designer at Comet Arcade. Pick the one twist from the menu that answers the brief. "
                      "Reply with the twist id on the first line, then its rules copied exactly."),
        HumanMessage(f"Brief : {brief}\n\nTwist menu :\n{MENU}"),
    ])
    board["twist"] = next((key for key in TWISTS if re.search(rf"\b{key}\b", reply.content.splitlines()[0].lower())), None)
    board["read"]["designer"] = reply.usage_metadata["input_tokens"]
    return reply.content


@tool
def ask_coder(rules: str, bugs: str = "") -> str:
    """Send the twist rules, and any bugs to fix, to the Coder. The Coder keeps the code on the project board."""
    board["order"].append("coder")
    board["attempts"] += 1
    ask = (f"{RULES}\n\n## The twist\n\n{rules}\n\n## The logic block\n\n```javascript\n{board['logic']}\n```\n\n"
           "Return the complete logic block with every rule and the twist in it, in one javascript code block. "
           "Keep every helper. Keep the function names init and step.")
    if bugs:
        ask += f"\n\n## Bugs to fix\n\n{bugs}"
    reply = crew_llm.invoke([
        SystemMessage("You are the Coder at Comet Arcade. You write the pure game logic of a one-file canvas game."),
        HumanMessage(ask),
    ])
    board["logic"] = code_block(reply.content)
    board["read"]["coder"] = reply.usage_metadata["input_tokens"]
    return f"Logic block saved to the board, {len(board['logic'].splitlines())} lines."


@tool
def run_game_tests() -> str:
    """Run the rule tests and the bot on the code on the project board. Returns the test report."""
    board["order"].append("tester")
    board["report"] = harness.run_tests(harness.splice(TEMPLATE, board["logic"]), board["twist"])
    return harness.render(board["report"])


producer = create_agent(crew_llm, tools=[ask_designer, ask_coder, run_game_tests], system_prompt=(
    "You are the Producer at Comet Arcade, a studio of agents. You take a brief and assign the work. "
    "First ask the Designer. Then send the Designer's rules to the Coder. Then run the tests. If tests fail, send the "
    "failures back to the Coder, at most 3 Coder attempts. Do one thing at a time. "
    "When the tests pass or the attempts run out, reply in two sentences : ship or do not ship, and why."))
print("brief:", BRIEFS[0]["brief"])
out = producer.invoke({"messages": [HumanMessage(BRIEFS[0]["brief"])]}, {"recursion_limit": 30})
print("crew order:", board["order"])
print("twist:", board["twist"])
print("logic lines on the board:", len(board["logic"].splitlines()))
print(harness.render(board["report"]))
print("producer calls, input tokens:", [m.usage_metadata["input_tokens"] for m in out["messages"] if m.type == "ai"])
print("crew, input tokens:", board["read"])
print("producer:", out["messages"][-1].content)

brief: Playtesters love Comet Snake but rage-quit when one comet ends a long run. Give them one way to survive.


crew order: ['designer', 'coder', 'tester']
twist: shield
logic lines on the board: 141
21 of 21 rule tests pass
bot : 51.2 stars on average, best score 97
producer calls, input tokens: [258, 456, 595, 644]
crew, input tokens: {'designer': 483, 'coder': 1891}
producer: Ship it: all 21 rule tests pass, and the bot averages 51.2 stars with a best score of 97. Comet Snake now offers a shield for one comet hit, earned through every fourth star.


## Things to try

Optional : we can change the game, the asset or the question and run the example again.